# Final results

Every trained CreditPFN checkpoint, every classical baseline,
and every untuned-TabPFN reference, scored on every held-out
test dataset with 5-fold cross-validation. The visualisations
here are the canonical "how did we do" view — used both for
thesis figures and for picking the headline checkpoint to
report.

## Where the data comes from

`scripts/eval_pipeline.py` writes one wide-format CSV per
(method × run × timestamp) under
`output/results/<TRACK>/<method-dirname>/...csv`.

Each row is one `(model × test_dataset × fold)` tuple with all
metric columns side-by-side:

* **PD (classification)** — `roc_auc, log_loss, pr_auc,
  optimal_threshold, f1, accuracy, precision, recall`.
* **LGD (regression)** — `rmse, mae, r2, neg_nll`.

Method directory names encode the architecture, and
`src/utils/eval_viz._decode_method_dirname` parses them into
structured `(source, base_short, lr, use_lora)` fields. The
human-readable label used in legends is built by
`human_method_name()`.

All loaders and plots live in `src/utils/eval_viz.py`;
this notebook only calls them.

## Direction of improvement

Higher-is-better metrics: `roc_auc`, `pr_auc`, `f1`, `accuracy`,
`precision`, `recall`, `r2`, `neg_nll`.
Lower-is-better: `rmse`, `mae`, `log_loss`. Every aggregation /
ranking helper is direction-aware.

## Sections

1. Setup and what's on disk
2. Headline leaderboard
3. Per-dataset performance
4. Pairwise comparisons (win-rate matrix, scatter)
5. Trained vs untuned TabPFN (the Real-TabPFN-style "did it help?" plot)
6. Stability and calibration
7. Time vs accuracy
8. Failures

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
REPO = Path(os.getcwd()).resolve()
while not (REPO / 'src' / 'utils' / 'eval_viz.py').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import pandas as pd
from src.utils.figures import open_figure_sink
from src.utils.eval_viz import (
    load_eval_results, available_methods, available_datasets,
    primary_metric, eval_leaderboard, failed_pairs,
    aggregate_per_method, aggregate_per_method_per_dataset, winrate_matrix,
    plot_leaderboard, plot_metric_boxplot, plot_per_dataset_heatmap,
    plot_winrate_matrix, plot_method_vs_method_scatter,
    plot_trained_vs_untuned, plot_baselines_vs_tabpfn,
    plot_fold_stability, plot_metric_correlation,
    plot_threshold_distribution, plot_top_method_per_dataset,
    plot_dataset_difficulty, plot_time_vs_metric,
)
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
TRACK = 'pd'    # 'pd' (classifier, ROC-AUC) or 'lgd' (regressor, RMSE)
sink = open_figure_sink('2.0_final_results')


## 1 · What's on disk

`load_eval_results` pools every CSV under
`output/results/<TRACK>/**/*.csv` and tags each row with
`(source, base_short, lr, use_lora, method_name, source_file)`
derived from the parent directory name.

If the tables below are empty, the eval pipeline hasn't produced
any CSVs yet for this track — the notebook will still render but
every plot will show a *(no data)* stub.

In [ ]:
print(f'Primary metric for track={TRACK}: {primary_metric(TRACK)}')
print(f'Methods on disk : {len(available_methods(TRACK))}')
print(f'Test datasets   : {len(available_datasets(TRACK))}')
available_methods(TRACK)

In [ ]:
raw = load_eval_results(TRACK)
print(f'rows: {len(raw)}')
if not raw.empty:
    display(raw.head(10))

## 2 · Headline leaderboard

Mean ± std of the primary metric (`roc_auc` for PD, `rmse` for
LGD) pooled over all `(dataset × fold)` rows of each method.
Sorted direction-aware so the best method is always at the top.

In [ ]:
eval_leaderboard(TRACK)

In [ ]:
sink.save(plot_leaderboard(TRACK), '01_leaderboard')

In [ ]:
sink.save(plot_metric_boxplot(TRACK), '02_metric_boxplot')

In [ ]:
sink.save(plot_baselines_vs_tabpfn(TRACK), '03_baselines_vs_tabpfn')

## 3 · Per-dataset performance

Each row of the heatmap is a method, each column a held-out test
dataset, each cell the mean of the primary metric over the
5-fold CV. The bottom plot "who wins on which dataset?" colours
each dataset by its best-performing method — a fast diagnostic
for finding datasets where the trained checkpoint isn't paying
off.

In [ ]:
sink.save(plot_per_dataset_heatmap(TRACK), '04_per_dataset_heatmap')

In [ ]:
aggregate_per_method_per_dataset(TRACK).round(3)

In [ ]:
sink.save(plot_top_method_per_dataset(TRACK), '05_top_method_per_dataset')

In [ ]:
sink.save(plot_dataset_difficulty(TRACK), '06_dataset_difficulty')

## 4 · Pairwise comparisons

The **win-rate matrix** counts, for each ordered pair of methods
(`row` vs `column`), the fraction of test datasets where `row`
beat `column` (direction-aware). It is the same plot type as
Fig. 3 of the [TabPFN-3 technical report](../papers/2026_Grinsztajn_et_al._TabPFN_3_Technical_Report.pdf) —
a fast way to see which method dominates which.

The **method-vs-method scatter** lets you pick any two methods
and view their per-dataset scores side by side; points above
the y = x line favour the y-axis method.

In [ ]:
winrate_matrix(TRACK).round(2)

In [ ]:
sink.save(plot_winrate_matrix(TRACK), '07_winrate_matrix')

In [ ]:
# Pick any two methods from `available_methods(TRACK)`. The defaults
# below compare XGBoost (the strongest classical) against the best
# trained CreditPFN (whichever currently leads the leaderboard).
methods = available_methods(TRACK)
if len(methods) >= 2:
    classical_pick = next((m for m in methods if m == 'xgboost'), methods[0])
    creditpfn_pick = next((m for m in eval_leaderboard(TRACK)['method_name'].tolist()
                           if m.startswith('trained (')), methods[-1])
    _ = plot_method_vs_method_scatter(TRACK, classical_pick, creditpfn_pick)
else:
    print('Need at least two methods on disk to draw a scatter.')

## 5 · Trained vs untuned TabPFN

The **Real-TabPFN** [Garg 2025](../papers/2025_Garg_et_al._Real_TabPFN_Improving_Tabular_Foundation_Models_via_Continued_Pre_training_With_Real_World_Data.pdf)
headline figure: for each `(test_dataset, trained_checkpoint)`,
scatter the trained metric against the matching untuned TabPFN
of the **same architecture**. Points above the y = x line are
datasets where continued pretraining actually helped.

Points are coloured by base architecture — so you can spot
*"v3 helps but v2.5 doesn't"* or vice versa.

In [ ]:
sink.save(plot_trained_vs_untuned(TRACK), '08_trained_vs_untuned')

## 6 · Stability and calibration

* **Fold stability** — std of the primary metric across the 5
  outer folds for each `(method, dataset)`. A method whose box
  sits high is reporting high variance — a worse choice for
  production even if its mean looks good.
* **Metric correlation** — pairwise correlation between every
  metric column on disk. Useful to know whether picking the
  ROC-AUC winner is robust to switching to PR-AUC, or whether
  there's a calibration / discrimination trade-off lurking.
* **Threshold distribution** *(PD only)* — F1-optimal threshold
  per method. A method whose threshold drifts far from 0.5
  produces poorly-calibrated probabilities.

In [ ]:
sink.save(plot_fold_stability(TRACK), '09_fold_stability')

In [ ]:
sink.save(plot_metric_correlation(TRACK), '10_metric_correlation')

In [ ]:
sink.save(plot_threshold_distribution(TRACK), '11_threshold_distribution')

## 7 · Time vs accuracy

Per-fold elapsed seconds against the primary metric. Pareto-dominated
points are methods that another method matches in less time.

In [ ]:
sink.save(plot_time_vs_metric(TRACK), '12_time_vs_metric')

## 8 · Failures

Any `(method, dataset, fold)` triple with status ≠ OK. Most often
an HPO timeout (Optuna budget exhausted) or a CUDA OOM on a wide
dataset. Investigate the `error` column before treating an empty
row in the heatmap as missing data.

In [ ]:
fails = failed_pairs(TRACK)
if fails.empty:
    print('No failed (method × dataset × fold) rows. ✔')
else:
    display(fails)